# Módulo correcciones de tipo de proceso
87 reglas cerradas que reparan rótulos de `tipo_proceso` rotos por la extracción
(palabras partidas o pegadas de filas vecinas). Cada regla exige tabla, cuadro,
páginas y literal exactos, y cuántas filas debe tocar. `tipo_proceso_extraido`
guarda lo que salió del PDF y `tipo_proceso` el rótulo publicado.

In [ ]:
import pandas as pd

ALCANCE_CUADRO = "correccion_acotada_por_cuadro"
ALCANCE_FILA_CONTEXTO = "correccion_acotada_por_fila_contexto"

# regla_id, tabla, cuadros, paginas, literal_extraido, literal_fuente, alcance, apariciones_fuente
REGLAS_TABLA = [
    ('R001', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [431, 432, 433, 436, 438],
     'CON CUR SAL ES CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 5),
    ('R002', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [429, 430, 434, 435, 437],
     'CONCURSALES CON CUR SAL ES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 5),
    ('R003', 'apelaciones_por_tipo_proceso', ['6.1.1.3'], [419, 420, 421, 422, 423, 424, 425, 426, 427, 428],
     'CONCURSALES CON SAL ES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 10),
    ('R004', 'apelaciones_por_tipo_proceso', ['5.1.1.3'], [153],
     'CONCURSALES LES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 1),
    ('R005', 'apelaciones_por_tipo_proceso', ['5.1.1', '5.1.1.3'], [143, 144, 145, 146, 147, 149, 150, 155, 156, 158],
     'CONCURSALES O',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 10),
    ('R006', 'apelaciones_por_tipo_proceso', ['5.1.2.3', '5.1.2.4', '6.1.2.3', '6.1.2.4'], [200, 214, 217, 469, 479, 480, 481, 482, 483, 484, 485, 487, 488],
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR DE',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 13),
    ('R007', 'apelaciones_por_tipo_proceso', ['5.1.2.3', '5.1.2.4', '6.1.2.4'], [199, 210, 211, 212, 213, 215, 216, 218, 219, 486],
     'DE CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 10),
    ('R008', 'apelaciones_por_tipo_proceso', ['5.1.2.4'], [220],
     'DE DESACUERDO DE LOS PADRES',
     'DESACUERDO DE LOS PADRES',
     'correccion_acotada_por_cuadro', 1),
    ('R009', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [429, 430, 431, 432, 433, 434, 436, 437, 438],
     'DE EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 9),
    ('R010', 'apelaciones_por_tipo_proceso', ['5.1.2.3', '6.1.2.3'], [201, 202, 203, 204, 205, 206, 207, 208, 209, 470, 471, 472, 473, 474, 475, 476, 477, 478],
     'DESACUERDO DE LOS PADRES DE',
     'DESACUERDO DE LOS PADRES',
     'correccion_acotada_por_cuadro', 18),
    ('R011', 'apelaciones_por_tipo_proceso', ['6.1.1.3'], [419, 420, 421, 422, 423, 424, 425, 426, 427, 428],
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO CUR DE IÓN',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 10),
    ('R012', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [435],
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO DE',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R013', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [435, 437],
     'IO ORDINARIO',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 2),
    ('R014', 'apelaciones_por_tipo_proceso', ['5.1.1', '5.1.1.3'], [148, 151, 152, 154, 157, 159, 160, 161, 162, 163, 164],
     'O CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 11),
    ('R015', 'apelaciones_por_tipo_proceso', ['5.1.1.3', '6.1.1.3', '6.1.1.4'], [143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 423, 424, 425, 426, 427, 429, 436],
     'O ORDINARIO',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 17),
    ('R016', 'apelaciones_por_tipo_proceso', ['5.1.1', '6.1.1.4'], [164, 438],
     'ORDINARIO O',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 2),
    ('R017', 'apelaciones_por_tipo_proceso', ['6.1.1.4'], [431, 433, 434],
     'RIO ORDINARIO',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 3),
    ('R018', 'causas_por_tipo_proceso', ['6.3.2.1'], [627],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE ANTICORRUPCIÒN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 1),
    ('R019', 'causas_por_tipo_proceso', ['5.3.2.1', '6.3.2.1'], [362, 363, 364, 625, 626, 627],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE PENAL COMUN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 16),
    ('R020', 'causas_por_tipo_proceso', ['6.3.2.1'], [627],
     'ACCIÓN PENAL PÙBLICA CONTRA LA',
     'ACCIÓN PENAL PÙBLICA',
     'correccion_acotada_por_cuadro', 1),
    ('R021', 'causas_por_tipo_proceso', ['5.3.2.1', '6.3.2.1'], [362, 363, 364, 625, 626, 627],
     'ANTICORRUPCIÒN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 20),
    ('R022', 'causas_por_tipo_proceso', ['5.2.1.1'], [305],
     'BENEFICIOS DEMANDA DE ADQUIRIDOS SOCIALES Y DERECHOS',
     'DEMANDA DE BENEFICIOS SOCIALES Y DERECHOS ADQUIRIDOS',
     'correccion_acotada_por_fila_contexto', 1),
    ('R023', 'causas_por_tipo_proceso', ['5.1.1.1'], [122],
     'CONCURSALES ESO LES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 1),
    ('R024', 'causas_por_tipo_proceso', ['6.1.1.1'], [399, 400, 401, 402, 403, 404, 405, 406, 407],
     'CONCURSALES S',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 9),
    ('R025', 'causas_por_tipo_proceso', ['6.1.1.1'], [408],
     'CONCURSALES SAL ES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 1),
    ('R026', 'causas_por_tipo_proceso', ['5.1.2.1', '6.1.2.1'], [178, 181, 186, 451],
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR DE',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 4),
    ('R027', 'causas_por_tipo_proceso', ['6.3.2.1'], [625, 626, 627],
     'CONTRA LA ACCIÓN PENAL PÙBLICA',
     'ACCIÓN PENAL PÙBLICA',
     'correccion_acotada_por_cuadro', 9),
    ('R028', 'causas_por_tipo_proceso', ['5.3.2.1'], [362, 363, 364],
     'CONTRA LA VIOLENCIA HACIA LA ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 10),
    ('R029', 'causas_por_tipo_proceso', ['5.1.2.1', '6.1.2.1'], [177, 179, 180, 182, 183, 184, 185, 187, 449, 450, 452, 453, 454, 455, 456, 457, 458],
     'DE CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 17),
    ('R030', 'causas_por_tipo_proceso', ['5.2.1.1'], [303, 308],
     'DEMANDA DE ADQUIRIDOS BENEFICIOS SOCIALES Y DERECHOS',
     'DEMANDA DE BENEFICIOS SOCIALES Y DERECHOS ADQUIRIDOS',
     'correccion_acotada_por_fila_contexto', 2),
    ('R031', 'causas_por_tipo_proceso', ['6.1.1.1'], [399, 400, 401, 402, 403, 404, 405, 406, 407],
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO DE',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 9),
    ('R032', 'causas_por_tipo_proceso', ['6.1.1.1'], [408],
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO S DE N',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R033', 'causas_por_tipo_proceso', ['6.1.1.1'], [405],
     'IO ORDINARIO',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 1),
    ('R034', 'causas_por_tipo_proceso', ['6.3.2.1'], [625, 626, 627],
     'LAS MUJERES ACCIÓN PENAL PRIVADA',
     'ACCIÓN PENAL PRIVADA',
     'correccion_acotada_por_cuadro', 10),
    ('R035', 'causas_por_tipo_proceso', ['5.1.1.1', '6.1.1.1'], [131, 408],
     'O ORDINARIO',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 2),
    ('R036', 'causas_por_tipo_proceso', ['6.1.1.1'], [399, 401, 403, 404],
     'PARTIDAS EN EL REGISTRO DE DERECHOS REALES, ASI COMO EN OTROS REGISTROS PÚBLICOS',
     'INSCRIPCIÓN, MODIFICACIÓN, CANCELACIÓN O FUSIÓN DE PARTIDAS EN EL REGISTRO DE DERECHOS REALES, ASI COMO EN OTROS REGISTROS PÚBLICOS',
     'correccion_acotada_por_fila_contexto', 4),
    ('R037', 'causas_por_tipo_proceso', ['5.3.2.1', '6.3.2.1'], [362, 363, 364, 627],
     'PENAL COMUN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 5),
    ('R038', 'causas_por_tipo_proceso', ['5.3.2.1'], [364],
     'PENAL PÙBLICA A INSTANCIA DE PARTE CONTRA LA VIOLENCIA HACIA LA ACCIÓN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 1),
    ('R039', 'causas_por_tipo_proceso', ['5.1.1.1'], [122, 124, 125, 127, 129, 130],
     'REGISTRO DE DERECHOS REALES, ASI COMO EN OTROS REGISTROS PÚBLICOS',
     'INSCRIPCIÓN, MODIFICACIÓN, CANCELACIÓN O FUSIÓN DE PARTIDAS EN EL REGISTRO DE DERECHOS REALES, ASI COMO EN OTROS REGISTROS PÚBLICOS',
     'correccion_acotada_por_fila_contexto', 6),
    ('R040', 'causas_por_tipo_proceso', ['6.1.1.1'], [399, 401, 403, 404],
     'TRADUCCIÓN DE DOCUMENTO EN IDIOMA EXTRANJERO INSCRIPCIÓN, MODIFICACIÓN, CANCELACIÓN O FUSIÓN DE',
     'TRADUCCIÓN DE DOCUMENTO EN IDIOMA EXTRANJERO',
     'correccion_acotada_por_fila_contexto', 4),
    ('R041', 'causas_por_tipo_proceso', ['5.1.1.1'], [122, 124, 125, 127, 129, 130],
     'TRADUCCIÓN DE DOCUMENTO EN IDIOMA EXTRANJERO INSCRIPCIÓN, MODIFICACIÓN, CANCELACIÓN O FUSIÓN DE PARTIDAS EN EL',
     'TRADUCCIÓN DE DOCUMENTO EN IDIOMA EXTRANJERO',
     'correccion_acotada_por_fila_contexto', 6),
    ('R042', 'causas_por_tipo_proceso', ['6.3.2.1'], [625, 626, 627],
     'VIOLENCIA HACIA ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 10),
    ('R043', 'ejecucion_por_tipo_proceso', ['5.1.1.5'], [165, 167, 168, 170, 171, 172, 173, 174, 175],
     'CONCURSALES O',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 9),
    ('R044', 'ejecucion_por_tipo_proceso', ['5.1.2.5', '6.1.2.5'], [221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 490, 491, 492, 493, 494, 495, 496, 497],
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR DE',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 19),
    ('R045', 'ejecucion_por_tipo_proceso', ['6.1.2.5'], [489, 498],
     'DE CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 2),
    ('R046', 'ejecucion_por_tipo_proceso', ['6.1.1.5'], [447],
     'DE EJE OS EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R047', 'ejecucion_por_tipo_proceso', ['6.1.1.5'], [445],
     'DE EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R048', 'ejecucion_por_tipo_proceso', ['6.1.1.5'], [444],
     'ION DE EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R049', 'ejecucion_por_tipo_proceso', ['6.1.1.5'], [439, 440, 441, 442, 443, 444, 445, 446, 447, 448],
     'LES CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 10),
    ('R050', 'ejecucion_por_tipo_proceso', ['5.1.1.5'], [166, 169],
     'O CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 2),
    ('R051', 'ejecucion_por_tipo_proceso', ['6.1.1.5'], [439, 440, 441, 442, 443, 446],
     'ON OS DE EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 6),
    ('R052', 'otros_tramites_por_tipo_proceso', ['6.1.2.7'], [502, 504],
     '415 Y SGTES. MODIFICACIÓN DE GUARDA',
     'MODIFICACIÓN DE GUARDA',
     'correccion_acotada_por_cuadro', 2),
    ('R053', 'otros_tramites_por_tipo_proceso', ['5.3.2.2', '6.3.2.2'], [367, 630],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE ANTICORRUPCIÒN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 3),
    ('R054', 'otros_tramites_por_tipo_proceso', ['5.3.2.2', '6.3.2.2'], [365, 366, 367, 628, 629, 630],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE PENAL COMUN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 19),
    ('R055', 'otros_tramites_por_tipo_proceso', ['5.3.2.2', '6.3.2.2'], [365, 366, 367, 628, 629, 630],
     'ANTICORRUPCIÒN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 18),
    ('R056', 'otros_tramites_por_tipo_proceso', ['6.1.2.7'], [502, 503, 504],
     'CESACIÓN DE LA ASISTENCIA FAMILIAR DEMANDAS NUEVAS DENTRO DE UN PROCESO ART.',
     'CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'correccion_acotada_por_cuadro', 8),
    ('R057', 'otros_tramites_por_tipo_proceso', ['5.1.2.6', '6.1.2.6'], [232, 233, 234, 499],
     'CESACIÓN DE LA ASISTENCIA FAMILIAR DENTRO DE UN',
     'CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'correccion_acotada_por_cuadro', 12),
    ('R058', 'otros_tramites_por_tipo_proceso', ['6.3.2.2'], [628, 629, 630],
     'CONTRA LA VIOLENCIA ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE HACIA LAS MUJERES',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 10),
    ('R059', 'otros_tramites_por_tipo_proceso', ['5.3.2.2'], [365, 366, 367],
     'CONTRA LA VIOLENCIA HACIA LAS ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 11),
    ('R060', 'otros_tramites_por_tipo_proceso', ['5.1.2.7'], [235, 236, 237, 238],
     'DEMANDAS NUEVAS DENTRO DE CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'correccion_acotada_por_cuadro', 11),
    ('R061', 'otros_tramites_por_tipo_proceso', ['6.1.2.7'], [502, 504],
     'DEMANDAS NUEVAS DENTRO DE UN PROCESO ART. CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'correccion_acotada_por_cuadro', 2),
    ('R062', 'otros_tramites_por_tipo_proceso', ['6.1.2.6'], [499, 500, 501],
     'DEMANDAS NUEVAS DISMINUCIÓN DE ASISTENCIA DE FAMILIA',
     'DISMINUCIÓN DE ASISTENCIA DE FAMILIA',
     'correccion_acotada_por_cuadro', 10),
    ('R063', 'otros_tramites_por_tipo_proceso', ['6.1.2.6'], [499, 500, 501],
     'DENTRO DE UN CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'CESACIÓN DE LA ASISTENCIA FAMILIAR',
     'correccion_acotada_por_cuadro', 9),
    ('R064', 'otros_tramites_por_tipo_proceso', ['5.1.2.6'], [232, 233, 234],
     'DISMINUCIÓN DE ASISTENCIA DE FAMILIA DEMANDAS NUEVAS',
     'DISMINUCIÓN DE ASISTENCIA DE FAMILIA',
     'correccion_acotada_por_cuadro', 11),
    ('R065', 'otros_tramites_por_tipo_proceso', ['6.1.2.7'], [502, 503, 504],
     'MODIFICACIÓN DE GUARDA 415 Y SGTES.',
     'MODIFICACIÓN DE GUARDA',
     'correccion_acotada_por_cuadro', 8),
    ('R066', 'otros_tramites_por_tipo_proceso', ['5.3.2.2'], [367],
     'PENAL COMUN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 2),
    ('R067', 'otros_tramites_por_tipo_proceso', ['5.1.2.6', '6.1.2.6'], [232, 233, 234, 499, 500, 501],
     'PROCESO ART. 415 Y MODIFICACIÓN DE GUARDA',
     'MODIFICACIÓN DE GUARDA',
     'correccion_acotada_por_cuadro', 21),
    ('R068', 'otros_tramites_por_tipo_proceso', ['5.1.2.6', '6.1.2.6'], [232, 233, 234, 499, 500, 501],
     'SGTES. MODIFICACIÓN DEL DERECHO A VISITAS',
     'MODIFICACIÓN DEL DERECHO A VISITAS',
     'correccion_acotada_por_cuadro', 21),
    ('R069', 'otros_tramites_por_tipo_proceso', ['5.1.2.7'], [235, 236, 237, 238],
     'UN PROCESO ART. 415 Y SGTES. MODIFICACIÓN DE GUARDA',
     'MODIFICACIÓN DE GUARDA',
     'correccion_acotada_por_cuadro', 11),
    ('R070', 'resueltas_por_tipo_proceso', ['5.3.2.3', '6.3.2.3'], [368, 369, 370, 633],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE ANTICORRUPCIÒN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 9),
    ('R071', 'resueltas_por_tipo_proceso', ['5.3.2.3'], [368, 369, 370],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE CONTRA LA VIOLENCIA HACIA LAS MUJERES',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 8),
    ('R072', 'resueltas_por_tipo_proceso', ['6.3.2.3'], [631, 632, 633],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE HACIA LAS MUJERES',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 9),
    ('R073', 'resueltas_por_tipo_proceso', ['5.3.2.3', '6.3.2.3'], [368, 369, 370, 631, 632, 633],
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE PENAL COMUN',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 13),
    ('R074', 'resueltas_por_tipo_proceso', ['6.3.2.3'], [631, 632, 633],
     'ACCIÓN PENAL PÙBLICA CONTRA LA VIOLENCIA',
     'ACCIÓN PENAL PÙBLICA',
     'correccion_acotada_por_cuadro', 9),
    ('R075', 'resueltas_por_tipo_proceso', ['5.3.2.3', '6.3.2.3'], [369, 370, 631, 632, 633],
     'ANTICORRUPCIÒN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 12),
    ('R076', 'resueltas_por_tipo_proceso', ['5.1.1.2'], [136, 137, 140, 141],
     'CONCURSALES O',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 4),
    ('R077', 'resueltas_por_tipo_proceso', ['5.1.2.2', '6.1.2.2'], [188, 459, 463],
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR DE',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 3),
    ('R078', 'resueltas_por_tipo_proceso', ['6.3.2.3'], [633],
     'CONTRA LA VIOLENCIA ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE HACIA LAS MUJERES',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 1),
    ('R079', 'resueltas_por_tipo_proceso', ['5.3.2.3'], [369, 370],
     'CONTRA LA VIOLENCIA HACIA LAS MUJERES ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 3),
    ('R080', 'resueltas_por_tipo_proceso', ['5.1.2.2', '6.1.2.2'], [189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 460, 461, 462, 464, 465, 466, 467, 468],
     'DE CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'CONSTITUCIÓN DEL PATRIMONIO FAMILIAR',
     'correccion_acotada_por_cuadro', 18),
    ('R081', 'resueltas_por_tipo_proceso', ['6.1.1.2'], [409, 411, 412, 413, 414, 415, 416, 417, 418],
     'DE EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 9),
    ('R082', 'resueltas_por_tipo_proceso', ['6.1.1.2'], [410],
     'DE N EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'EJECUCIÓN COACTIVA DE SUMAS DE DINERO',
     'correccion_acotada_por_cuadro', 1),
    ('R083', 'resueltas_por_tipo_proceso', ['5.1.1.2'], [132, 134, 135, 138, 139, 142],
     'O CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 6),
    ('R084', 'resueltas_por_tipo_proceso', ['5.1.1.2'], [142],
     'ORDINARIO O',
     'ORDINARIO',
     'correccion_acotada_por_cuadro', 1),
    ('R085', 'resueltas_por_tipo_proceso', ['6.1.1.2'], [415],
     'OTROS REGISTROS PÚBLICOS',
     'INSCRIPCIÓN, MODIFICACIÓN, CANCELACIÓN O FUSIÓN DE PARTIDAS EN EL REGISTRO DE DERECHOS REALES, ASI COMO EN OTROS REGISTROS PÚBLICOS',
     'correccion_acotada_por_fila_contexto', 1),
    ('R086', 'resueltas_por_tipo_proceso', ['5.3.2.3', '6.3.2.3'], [369, 370, 631, 632, 633],
     'PENAL COMUN ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'ACCIÓN PENAL PÙBLICA A INSTANCIA DE PARTE',
     'correccion_acotada_por_cuadro', 8),
    ('R087', 'resueltas_por_tipo_proceso', ['5.1.1.2'], [133],
     'SO CONCURSALES',
     'CONCURSALES',
     'correccion_acotada_por_cuadro', 1),
]

CORRECCIONES_TIPO_PROCESO = []
for fila in REGLAS_TABLA:
    CORRECCIONES_TIPO_PROCESO.append({
        "regla_id": fila[0],
        "tabla": fila[1],
        "cuadros": fila[2],
        "paginas": fila[3],
        "literal_extraido": fila[4],
        "literal_fuente": fila[5],
        "alcance": fila[6],
        "apariciones_fuente": fila[7],
    })

In [ ]:
FILAS_FUENTE_ESPERADAS = {
    "causas_por_tipo_proceso": 151,
    "resueltas_por_tipo_proceso": 116,
    "apelaciones_por_tipo_proceso": 128,
    "ejecucion_por_tipo_proceso": 51,
    "otros_tramites_por_tipo_proceso": 189,
}
FILAS_FISICAS_ESPERADAS = {
    "causas_por_tipo_proceso": 151,
    "resueltas_por_tipo_proceso": 1486,
    "apelaciones_por_tipo_proceso": 1194,
    "ejecucion_por_tipo_proceso": 255,
    "otros_tramites_por_tipo_proceso": 1194,
}
DOMINIOS_ANTES = {
    "causas_por_tipo_proceso": 130,
    "resueltas_por_tipo_proceso": 125,
    "apelaciones_por_tipo_proceso": 137,
    "ejecucion_por_tipo_proceso": 113,
    "otros_tramites_por_tipo_proceso": 49,
}
DOMINIOS_DESPUES = {
    "causas_por_tipo_proceso": 107,
    "resueltas_por_tipo_proceso": 109,
    "apelaciones_por_tipo_proceso": 121,
    "ejecucion_por_tipo_proceso": 106,
    "otros_tramites_por_tipo_proceso": 34,
}

In [ ]:
def reglas_para_tabla(tabla):
    if tabla not in FILAS_FUENTE_ESPERADAS:
        raise ValueError("Tabla desconocida: " + tabla)
    reglas = []
    for r in CORRECCIONES_TIPO_PROCESO:
        if r["tabla"] == tabla:
            reglas.append(r)
    return reglas


def reglas_expandidas():
    # Una entrada por (regla, cuadro).
    salida = []
    for r in CORRECCIONES_TIPO_PROCESO:
        for cuadro in r["cuadros"]:
            salida.append((r["regla_id"], r["tabla"], cuadro, r["literal_extraido"], r["literal_fuente"], r["alcance"]))
    return salida


def mascara_regla(df, regla, columna_literal):
    mascara = df["cuadro_origen"].astype(str).isin(regla["cuadros"]) & (df[columna_literal] == regla["literal_extraido"])
    if regla["alcance"] == ALCANCE_FILA_CONTEXTO:
        mascara = mascara & pd.to_numeric(df["pagina_pdf"], errors="coerce").isin(regla["paginas"])
    return mascara


def contar_coincidencias_por_fila(df, tabla, columna_literal):
    conteos = pd.Series(0, index=df.index, dtype="Int64")
    for regla in reglas_para_tabla(tabla):
        conteos = conteos + mascara_regla(df, regla, columna_literal).astype("Int64")
    return conteos


def incorporar_correcciones_tipo_proceso(df, tabla):
    requeridas = {"cuadro_origen", "pagina_pdf", "orden_fila", "tipo_proceso"}
    faltantes = requeridas - set(df.columns)
    if len(faltantes) > 0:
        raise ValueError("Faltan columnas para corregir tipo_proceso: " + str(sorted(faltantes)))
    if "tipo_proceso_extraido" in df.columns:
        raise ValueError("tipo_proceso_extraido ya existe; se evita una doble corrección")

    resultado = df.copy()
    posicion = resultado.columns.get_loc("tipo_proceso")
    resultado.insert(posicion, "tipo_proceso_extraido", resultado["tipo_proceso"].copy())
    aplicaciones = pd.Series(0, index=resultado.index, dtype="Int64")

    for regla in reglas_para_tabla(tabla):
        mascara = mascara_regla(resultado, regla, "tipo_proceso_extraido")
        observadas = int(mascara.sum())
        if observadas != regla["apariciones_fuente"]:
            raise ValueError(regla["regla_id"] + ": se esperaban " + str(regla["apariciones_fuente"])
                             + " filas fuente y se observaron " + str(observadas))
        filas = resultado.loc[mascara]
        cuadros_observados = set(filas["cuadro_origen"].astype(str))
        if cuadros_observados != set(regla["cuadros"]):
            raise ValueError(regla["regla_id"] + ": cuadros inesperados " + repr(cuadros_observados))
        paginas_observadas = set(pd.to_numeric(filas["pagina_pdf"], errors="raise").astype(int))
        if paginas_observadas != set(regla["paginas"]):
            raise ValueError(regla["regla_id"] + ": páginas inesperadas " + repr(paginas_observadas))
        if regla["alcance"] == ALCANCE_FILA_CONTEXTO:
            fuera = (resultado["cuadro_origen"].astype(str).isin(regla["cuadros"])
                     & (resultado["tipo_proceso_extraido"] == regla["literal_extraido"])
                     & ~pd.to_numeric(resultado["pagina_pdf"], errors="coerce").isin(regla["paginas"]))
            if fuera.any():
                raise ValueError(regla["regla_id"] + ": literal también observado fuera de las páginas auditadas")
        aplicaciones = aplicaciones + mascara.astype("Int64")
        resultado.loc[mascara, "tipo_proceso"] = regla["literal_fuente"]

    dobles = int((aplicaciones > 1).sum())
    if dobles > 0:
        raise ValueError(str(dobles) + " filas fuente reciben más de una regla")
    observadas = int((aplicaciones == 1).sum())
    esperadas = FILAS_FUENTE_ESPERADAS[tabla]
    if observadas != esperadas:
        raise ValueError(tabla + ": se esperaban " + str(esperadas) + " filas corregidas y se observaron " + str(observadas))
    return resultado

In [ ]:
# Chequeo del contrato: 87 reglas, 80 por cuadro y 7 por fila, sin claves repetidas.
ids = []
alcances = []
for r in CORRECCIONES_TIPO_PROCESO:
    ids.append(r["regla_id"])
    alcances.append(r["alcance"])
assert len(CORRECCIONES_TIPO_PROCESO) == 87
assert len(ids) == len(set(ids))
assert alcances.count(ALCANCE_CUADRO) == 80
assert alcances.count(ALCANCE_FILA_CONTEXTO) == 7
claves = []
for r in reglas_expandidas():
    claves.append((r[1], r[2], r[3]))
assert len(claves) == len(set(claves))